# Practical pandas cheat sheet for large datasets

Short, targeted patterns for everyday analysis. Assume the main DataFrame is named `data`. Prefer vectorized operations and the most specific indexer that fits the job.

In [ ]:
import numpy as np
import pandas as pd

## 1. Quick inspection

Inspect shape, dtypes, memory, and a small sample before doing expensive work.

In [ ]:
data.shape                         # (rows, columns)
data.head()                        # first 5 rows
data.sample(5, random_state=1)     # random rows
data.dtypes
data.info(memory_usage='deep')
data.memory_usage(deep=True).sort_values(ascending=False)

## 2. Selection and indexing: `loc`, `iloc`, `at`, `iat`

- `.loc[...]` uses **labels**; `.iloc[...]` uses **integer positions**.
- `.at[row_label, col_label]` and `.iat[row_pos, col_pos]` target exactly **one cell** and are the clearest fast scalar accessors.
- A scalar row selector returns a **Series**; a list-like row selector returns a **DataFrame**.
- On very large data, `data.loc[idx]` can be far quicker than `data.loc[[idx]]` because the latter performs list-like selection and constructs a 2-D result.

In [ ]:
data.loc[idx]                       # one label -> Series (unique index)
data.loc[[idx]]                     # list of labels -> DataFrame
data.loc[idx, 'arrival_delay']      # scalar when labels are unique
data.loc[[idx], ['airline']]        # always 2-D DataFrame

data.iloc[0]                        # first row -> Series
data.iloc[[0]]                      # first row -> DataFrame
data.iloc[:100, [0, 3]]             # positional slice

data.at[idx, 'arrival_delay']       # one cell by label
data.iat[0, 3]                      # one cell by position

## 3. Boolean masks

Masks are vectorized and expressive, but they normally inspect every row: expect an **O(n) scan**. Build a mask once if you will reuse it. Use `&`, `|`, and `~`—not Python's `and`, `or`, and `not`—and parenthesize each condition.

In [ ]:
mask = (data['airline'].eq('VX')) & (data['arrival_delay'].gt(60))
late_vx = data.loc[mask, ['flight_number', 'arrival_delay']]

data.loc[data['state'].isin(['MA', 'NY'])]
data.loc[data['name'].str.contains('smith', case=False, na=False)]
data.loc[~mask]                      # invert a mask

n_matches = mask.sum()              # True values
match_rate = mask.mean()            # fraction True

## 4. Null handling

Use `.isna()` / `.notna()`; comparisons such as `value == np.nan` do not work. `count()` excludes nulls, while `size` and `len` include them.

In [ ]:
data.loc[data['arrival_delay'].isna()]
data.loc[data[['origin', 'destination']].isna().any(axis=1)]  # any null
data.loc[data[['origin', 'destination']].isna().all(axis=1)]  # all null

data.isna().sum().sort_values(ascending=False)
data['arrival_delay'].fillna(0)
data.dropna(subset=['origin', 'destination'])

len(data)                  # number of rows, nulls included
data['col'].size           # elements, nulls included
data['col'].count()        # non-null elements
data.size                  # total cells = rows * columns

## 5. Assignment

Assign in one `.loc[...]` operation to avoid chained-assignment bugs. For one known cell, use `.at`. Vectorized assignments align on index labels.

In [ ]:
data.at[idx, 'status'] = 'review'                  # one cell
data.loc[mask, 'status'] = 'late'                  # matching rows
data.loc[mask, ['flag', 'score']] = [True, 1]      # multiple columns
data['delay_hours'] = data['arrival_delay'].div(60)

# Series assignment aligns by index label, even if order differs.
data['normalized'] = other_series

# Use positional assignment only when deliberately intended.
data['normalized'] = other_series.to_numpy()

# Avoid: data[data['x'] > 0]['flag'] = True  # chained assignment

## 6. Type conversion

`errors='coerce'` replaces invalid numeric/datetime values with nulls. Nullable extension dtypes retain missing values.

In [ ]:
data['amount'] = pd.to_numeric(data['amount'], errors='coerce')
data['count'] = data['count'].astype('Int32')       # nullable integer
data['active'] = data['active'].astype('boolean')   # nullable boolean
data['code'] = data['code'].astype('string')        # pandas string dtype
data['airline'] = data['airline'].astype('category') # repeated labels

data = data.convert_dtypes()
data.dtypes

## 7. Datetime and timedelta operations

Parse once, then use vectorized `.dt` operations. Supply a known `format=` when possible—it is clearer and often faster.

In [ ]:
data['departed_at'] = pd.to_datetime(
    data['departed_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce'
)
data['date'] = data['departed_at'].dt.date
data['year'] = data['departed_at'].dt.year
data['weekday'] = data['departed_at'].dt.day_name()
data['month'] = data['departed_at'].dt.to_period('M')

data['duration'] = data['arrived_at'] - data['departed_at']
data['duration_minutes'] = data['duration'].dt.total_seconds().div(60)
data['departed_at'] + pd.Timedelta(hours=2)

recent = data.loc[data['departed_at'].between('2025-01-01', '2025-01-31')]

## 8. Grouping and `groupby`

Use named aggregation for readable columns. `GroupBy.size()` counts rows including null values; `GroupBy.count()` counts non-null values in selected columns. `transform()` returns one result per original row and preserves the original index.

In [ ]:
summary = (
    data.groupby('airline', observed=True)
        .agg(
            flights=('flight_number', 'size'),
            known_delays=('arrival_delay', 'count'),
            mean_delay=('arrival_delay', 'mean'),
            max_delay=('arrival_delay', 'max'),
        )
        .sort_values('flights', ascending=False)
)

rows_per_group = data.groupby('airline', observed=True).size()
non_null_per_group = data.groupby('airline', observed=True)['arrival_delay'].count()

# Same length/index as data: safe label-aligned assignment.
data['airline_mean_delay'] = (
    data.groupby('airline', observed=True)['arrival_delay'].transform('mean')
)

## 9. Duplicates

In [ ]:
data.duplicated().sum()
data.loc[data.duplicated(['flight_id'], keep=False)]  # show every duplicate key

deduped = data.drop_duplicates(['flight_id'], keep='last')
unique_keys = data['flight_id'].nunique(dropna=False)

data.index.is_unique
data.index.duplicated().sum()

## 10. Sorting and top/bottom values

Sorting every row is roughly O(n log n). If you need only a few extremes, prefer `nlargest` / `nsmallest`.

In [ ]:
data.sort_values(['airline', 'arrival_delay'], ascending=[True, False])
data.sort_index()
data.nlargest(10, 'arrival_delay')
data.nsmallest(10, 'arrival_delay')

## 11. Aggregation and counts

In [ ]:
data['arrival_delay'].agg(['count', 'mean', 'median', 'min', 'max'])
data[['arrival_delay', 'departure_delay']].describe()
data['airline'].value_counts(dropna=False)
data['airline'].value_counts(normalize=True, dropna=False)
data['airline'].nunique(dropna=False)
pd.crosstab(data['airline'], data['cancelled'], normalize='index')

## 12. String operations

Use `.str` methods for vectorized cleanup. Set `na=False` for Boolean searches when nulls should not match.

In [ ]:
data['city'] = data['city'].str.strip().str.title()
data['code'] = data['code'].str.upper()
data['is_boston'] = data['city'].str.contains('Boston', case=False, na=False)
data[['prefix', 'number']] = data['flight_code'].str.extract(r'([A-Z]+)(\d+)')
data['route'].str.split('-', n=1, expand=True)
data['notes'].str.replace(r'\s+', ' ', regex=True)

## 13. Efficient lookups and joins

A unique index makes repeated label lookups natural. Avoid repeatedly scanning a column for the same key. For simple substitution, `.map()` is usually clearer than a row-wise function.

In [ ]:
by_flight = data.set_index('flight_id', verify_integrity=True)
row = by_flight.loc[flight_id]                 # Series for one unique label
value = by_flight.at[flight_id, 'arrival_delay']

carrier_name = {'VX': 'Virgin America', 'AA': 'American'}
data['carrier_name'] = data['airline'].map(carrier_name)

enriched = data.merge(airports, on='airport_code', how='left', validate='m:1')
# validate='m:1' catches duplicate keys in the lookup table.

## 14. Memory and performance tips

1. Select only the columns and rows you need; avoid copying a huge DataFrame unnecessarily.
2. Prefer vectorized arithmetic, comparisons, `.str`, `.dt`, `.map`, `groupby`, and `merge` over Python loops or `axis=1` `.apply()`.
3. Use compact dtypes (`category`, nullable integers, smaller numeric widths) when their ranges/cardinality fit.
4. Parse and convert once, not inside repeated filters or group operations.
5. Reuse masks and indexed lookup tables when querying repeatedly.
6. Read less data: choose columns, filter early, and use chunking when the full dataset cannot fit in memory. Parquet is usually better than CSV for repeated analytical reads.
7. Measure on representative data—clear-looking code is not always the bottleneck.

In [ ]:
# Read only what you need.
cols = ['flight_id', 'airline', 'arrival_delay']
data = pd.read_csv('flights.csv', usecols=cols, dtype={'airline': 'category'})

# Process a CSV that does not fit in memory.
for chunk in pd.read_csv('flights.csv', usecols=cols, chunksize=1_000_000):
    result = chunk.loc[chunk['arrival_delay'].gt(60)]
    # aggregate/write each result instead of collecting every chunk in RAM

memory_mb = data.memory_usage(deep=True).sum() / 1024**2

## 15. Common gotchas

- **Labels vs positions:** `.loc[5]` means index label `5`; `.iloc[5]` means the sixth row.
- **Inclusive label slices:** `data.loc['a':'c']` includes both endpoints; ordinary Python/`.iloc` slices exclude the stop.
- **Series vs DataFrame:** `data.loc[idx]` generally returns a Series; `data.loc[[idx]]` returns a DataFrame. A non-unique index can make scalar-label `.loc[idx]` return multiple rows.
- **Index alignment:** pandas aligns Series/DataFrames by labels during arithmetic and assignment. Use `.to_numpy()` only when positional behavior is truly intended.
- **Null equality:** use `.isna()`, never `== np.nan`.
- **Boolean logic:** use `(a) & (b)`, `(a) | (b)`, and `~a`; parentheses are required.
- **Membership:** use `series.isin(values)`, not `series in values`.
- **Chained assignment:** use `data.loc[mask, 'col'] = value`.
- **`size` vs `len` vs `count`:** `len(data)` is rows, `data.size` is all cells, and `Series.count()` excludes nulls. `GroupBy.size()` counts rows; `GroupBy.count()` counts non-nulls per column.
- **`inplace=True`:** it is not guaranteed to save memory or time; explicit reassignment is often easier to reason about.
- **`apply(axis=1)`:** convenient but often slow on large data; look for a vectorized expression first.
- **Silent join explosions:** use `merge(..., validate='m:1')`, `'1:1'`, etc. when you know the expected key relationship.

## Quick choice guide

| Goal | Preferred pattern | Typical result |
|---|---|---|
| One cell by label | `data.at[idx, col]` | scalar |
| One cell by position | `data.iat[i, j]` | scalar |
| One row by label | `data.loc[idx]` | Series (unique index) |
| One row, keep 2-D | `data.loc[[idx]]` | DataFrame |
| Several rows by position | `data.iloc[positions]` | DataFrame |
| Rows matching conditions | `data.loc[mask, columns]` | DataFrame/Series |
| Per-group summary | `groupby(...).agg(...)` | reduced result |
| Per-row group statistic | `groupby(...).transform(...)` | Series aligned to original index |
| Repeated key lookup | unique index + `.loc`/`.at` | Series/scalar |